# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Out-of-fold predictions and confusion analysis
# ------------------------------
# Verified best configuration from the preceding sweeps.
BEST_CHAR_RANGE = (1, 5)
BEST_C = 2.0

from sklearn.metrics import confusion_matrix, classification_report

classes = np.unique(labels)
oof_predictions = np.empty(labels.shape, dtype=object)
oof_margin = np.full(labels.shape, np.nan, dtype=np.float32)
fold_records = []
experiment_started = time.time()

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    y_train, y_valid = labels[train_idx], labels[valid_idx]

    word_vectorizer = make_word_vectorizer()
    X_word_train = word_vectorizer.fit_transform(texts[train_idx])
    X_word_valid = word_vectorizer.transform(texts[valid_idx])

    punctuation_scaler = StandardScaler()
    X_punct_train = csr_matrix(punctuation_scaler.fit_transform(punctuation[train_idx]))
    X_punct_valid = csr_matrix(punctuation_scaler.transform(punctuation[valid_idx]))

    char_vectorizer = make_char_vectorizer(BEST_CHAR_RANGE)
    X_char_train = char_vectorizer.fit_transform(texts[train_idx])
    X_char_valid = char_vectorizer.transform(texts[valid_idx])

    X_train = hstack([X_word_train, X_char_train, X_punct_train], format='csr')
    X_valid = hstack([X_word_valid, X_char_valid, X_punct_valid], format='csr')

    model = LinearSVC(C=BEST_C, class_weight='balanced', max_iter=MAX_ITER)
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    decision = model.decision_function(X_valid)
    sorted_scores = np.sort(decision, axis=1)

    oof_predictions[valid_idx] = predictions
    oof_margin[valid_idx] = sorted_scores[:, -1] - sorted_scores[:, -2]
    fold_score = float(balanced_accuracy_score(y_valid, predictions))
    fold_records.append({'fold': fold_number, 'balanced_accuracy': fold_score, 'validation_rows': int(len(valid_idx))})
    print(f'Fold {fold_number}/{N_SPLITS}: balanced_accuracy={fold_score:.4f}')

# Every row must have exactly one out-of-fold prediction.
assert not pd.isna(oof_predictions).any(), 'OOF predictions are incomplete'
oof_score = float(balanced_accuracy_score(labels, oof_predictions))
cm = confusion_matrix(labels, oof_predictions, labels=classes)
report = classification_report(labels, oof_predictions, labels=classes, output_dict=True, zero_division=0)

# Rank directed confusion pairs by number of OOF errors.
pair_rows = []
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        if i != j and cm[i, j] > 0:
            pair_rows.append({'actual': actual, 'predicted': predicted, 'count': int(cm[i, j])})
pair_rows.sort(key=lambda row: row['count'], reverse=True)

per_class = []
for cls in classes:
    stats = report[str(cls)]
    per_class.append({
        'model': cls,
        'precision': float(stats['precision']),
        'recall': float(stats['recall']),
        'f1_score': float(stats['f1-score']),
        'support': int(stats['support']),
    })
per_class.sort(key=lambda row: row['recall'])

# Save machine-readable outputs for later pairwise/specialist experiments.
output_dir = Path('/kaggle/working')
summary = {
    'experiment': 'OOF confusion analysis',
    'split_method': '5-fold GroupKFold by prompt_id',
    'model_config': {
        'word_ngram_range': list(WORD_NGRAM_RANGE),
        'char_ngram_range': list(BEST_CHAR_RANGE),
        'punctuation': 'all character frequency features',
        'linear_svc_C': BEST_C,
    },
    'oof_balanced_accuracy': oof_score,
    'fold_records': fold_records,
    'classes': classes.tolist(),
    'confusion_matrix': cm.tolist(),
    'top_confusion_pairs': pair_rows[:30],
    'lowest_recall_models': per_class,
    'total_elapsed_seconds': round(time.time() - experiment_started, 2),
}
(output_dir / 'oof_confusion_results.json').write_text(json.dumps(summary, indent=2))
pd.DataFrame({'actual': labels, 'predicted': oof_predictions, 'margin': oof_margin, 'prompt_id': groups}).to_csv(output_dir / 'oof_predictions.csv', index=False)
pd.DataFrame(pair_rows).to_csv(output_dir / 'confusion_pairs.csv', index=False)
pd.DataFrame(per_class).to_csv(output_dir / 'per_class_report.csv', index=False)

print(f'OOF balanced accuracy: {oof_score:.4f}')
print('\\nTop confusion pairs:')
for row in pair_rows[:15]:
    print(f"{row['actual']} -> {row['predicted']}: {row['count']}")
print('\\nLowest recall models:')
for row in per_class[:10]:
    print(f"{row['model']}: recall={row['recall']:.4f}, support={row['support']}")
print('Saved OOF analysis artifacts to /kaggle/working/')


## OOF analysis interpretation

OOF predictions are generated only by models that did not train on the corresponding rows. The confusion pairs and low-recall classes will guide any later pairwise specialist or ensemble experiment.
